# Certified Flexibility Orchestration — Experiments (Google Colab)

**Companion notebook to:** Zayoud, Guizani, Alghamdi and Hamam,
*Certified Flexibility Orchestration for Grid-Responsive Buildings with Electric
Vehicles: Selective Commitment and the Reliability–Mobility Trade-off*,
submitted to **Energy Nexus**, Special Issue *Grid-responsive Energy Flexible
Buildings with Electric Vehicles for Low-carbon Transitions*.

---

## How to run

1. `Runtime → Run all`. A CPU runtime is sufficient; no GPU is used.
2. Authorise Google Drive when prompted (Section 1).
3. Everything is written to **`MyDrive/CFO_GridResponsive/`** — figures to
   `figs/`, result tables to `out/`, and the generated library to `lib/`.
   Re-running the notebook overwrites those outputs in place.

**Runtime.** `DAYS = 42` takes roughly 45–60 minutes on a standard Colab CPU
runtime. Set `QUICK = True` in Section 2 for a ~10 minute smoke run that
reproduces the structure but not the statistics.

**Scope.** All exogenous series — weather, irradiance, occupancy, marginal grid
carbon intensity, tariff and EV mobility — are *synthetic*, generated from
physically-grounded but stipulated models. Each generator sits behind a narrow
interface (`Scenario._make_*`) so measured traces can be substituted without
touching the controllers or the certification layer. Nothing here is field
validation.

## 1. Environment and Google Drive

In [ ]:
#@title Install dependencies and mount Google Drive { display-mode: "form" }
import sys, subprocess, os

IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)

# cvxpy is not pre-installed on Colab; everything else is.
try:
    import cvxpy  # noqa: F401
    print("cvxpy already available:", cvxpy.__version__)
except ImportError:
    print("installing cvxpy ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cvxpy"],
                   check=True)
    import cvxpy
    print("cvxpy installed:", cvxpy.__version__)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
else:
    DRIVE_ROOT = os.path.expanduser("~")     # local fallback
    print("Not in Colab - writing to", DRIVE_ROOT)

In [ ]:
#@title Project folders on Drive { display-mode: "form" }
from pathlib import Path

PROJECT_NAME = "CFO_GridResponsive"  #@param {type:"string"}

PROJECT = Path(DRIVE_ROOT) / PROJECT_NAME
FIGS    = PROJECT / "figs"
OUT     = PROJECT / "out"
LIBDIR  = PROJECT / "lib"
for p in (PROJECT, FIGS, OUT, LIBDIR):
    p.mkdir(parents=True, exist_ok=True)

print("project :", PROJECT)
print("figures :", FIGS)
print("tables  :", OUT)

In [ ]:
#@title Write the simulation library to Drive and import it { display-mode: "form" }
# The library is written to Drive so that it persists across sessions and can be
# edited there directly. Expand this cell only if you need to inspect the source.
LIB_SOURCE = r'''
"""
Certified Flexibility Orchestration (CFO) -- experimental engine.

Reference implementation supporting the manuscript
"Certified Flexibility Orchestration for Grid-Responsive Buildings with
Electric Vehicles: A Unified Variational Framework with Distribution-Free
Reliability Guarantees".

All exogenous series are generated by physically-grounded synthetic models so
that the notebook is self-contained and reproducible without data licences.
Every generator is isolated behind a small interface (see `Scenario`) so that
real weather / carbon / mobility traces can be substituted without touching
the controllers.
"""

from __future__ import annotations

import numpy as np
import pandas as pd
import cvxpy as cp
from dataclasses import dataclass, field

# ----------------------------------------------------------------------------
# 1. Configuration
# ----------------------------------------------------------------------------


@dataclass
class BuildingCfg:
    """Single-zone RC building with HVAC, PV and stationary storage."""
    name: str = "CaseA"
    # thermal
    R: float = 4.5e-3          # K/W  envelope resistance
    C: float = 2.2e8           # J/K  effective thermal capacitance
    hvac_max: float = 180.0    # kW electrical
    cop_cool: float = 3.2
    cop_heat: float = 3.0
    theta_lo: float = 21.0     # comfort band, degC
    theta_hi: float = 24.5
    theta_set: float = 22.75
    # internal / solar gains
    gain_occ_kw: float = 60.0
    sol_gain_frac: float = 0.28
    # electrical
    base_load_kw: float = 165.0
    pv_kwp: float = 180.0
    # stationary storage
    bat_kwh: float = 180.0
    bat_kw: float = 75.0
    bat_eff: float = 0.95
    bat_deg_cost: float = 0.055     # $/kWh throughput
    bat_soc0: float = 0.5
    # climate regime
    winter_peaking: bool = False
    lat_seasonal_amp: float = 11.0
    mean_temp: float = 14.0


@dataclass
class FleetCfg:
    """EV fleet described as behavioural clusters."""
    n_clusters: int = 3
    cluster_sizes: tuple = (10, 8, 5)
    batt_kwh: float = 60.0
    charger_kw: float = 11.0
    v2b_eff: float = 0.93
    deg_cost: float = 0.075          # $/kWh throughput (higher: user asset)
    # behaviour, per cluster: (arrive_mu, depart_mu, soc_arr_mu, soc_req, p_part)
    profiles: tuple = (
        (8.0, 17.5, 0.55, 0.80, 0.85),   # regular commuters
        (7.5, 16.0, 0.45, 0.70, 0.70),   # early leavers
        (9.5, 19.0, 0.60, 0.85, 0.55),   # late / low participation
    )
    arrive_sd: float = 0.9
    depart_sd: float = 1.1
    early_hazard: float = 0.18       # P(leaves earlier than declared)
    early_shift_mu: float = 2.2      # hours earlier, when it happens


@dataclass
class SimCfg:
    seed: int = 20261231
    days: int = 56                   # representative days
    dt: float = 1.0                  # hours
    horizon: int = 24                # MPC prediction horizon
    forecast_severity: float = 1.0   # scales all forecast error
    n_events_per_day: float = 2.2    # grid service requests
    nd_penalty: float = 0.45         # $/kWh of undelivered promise
    alpha: float = 0.10              # nominal miscoverage
    calib_min: int = 18              # min residuals before certifying
    adaptive_lr: float = 0.02        # adaptive conformal step
    export_price_frac: float = 0.35
    deliver_tol: float = 0.05        # settlement tolerance on delivery


# ----------------------------------------------------------------------------
# 2. Exogenous scenario generation
# ----------------------------------------------------------------------------


class Scenario:
    """Generates weather, PV, load, carbon, tariff and EV mobility.

    Replace the individual `_make_*` methods to plug in measured traces.
    """

    def __init__(self, bcfg: BuildingCfg, fcfg: FleetCfg, scfg: SimCfg):
        self.b, self.f, self.s = bcfg, fcfg, scfg
        self.rng = np.random.default_rng(scfg.seed)
        self.T = int(scfg.days * 24 / scfg.dt)
        self.hour = np.arange(self.T) % 24
        self.day = np.arange(self.T) // 24
        # sample representative days spread across the year
        self.doy = np.repeat(
            np.linspace(5, 360, scfg.days).astype(int), 24)
        self._build()

    # -- individual generators -------------------------------------------------
    def _make_ambient(self):
        b = self.b
        seasonal = b.mean_temp + b.lat_seasonal_amp * np.cos(
            2 * np.pi * (self.doy - 200) / 365.0)
        diurnal = 4.5 * np.sin(2 * np.pi * (self.hour - 9) / 24.0)
        noise = self._ar1(0.86, 1.5)
        return seasonal + diurnal + noise

    def _make_solar(self):
        elev = np.maximum(
            0.0,
            np.sin(np.pi * (self.hour - 6.5) / 11.5)
            * (0.72 + 0.28 * np.cos(2 * np.pi * (self.doy - 172) / 365.0)),
        )
        # daily cloud factor, persistent within a day
        cloud_day = self.rng.beta(5.0, 1.8, size=self.s.days)
        cloud = np.repeat(cloud_day, 24)
        intra = np.clip(1.0 + 0.18 * self._ar1(0.7, 1.0), 0.35, 1.3)
        return np.clip(elev * cloud * intra, 0, None)

    def _make_carbon(self, pv_norm):
        """Marginal grid carbon intensity, kgCO2/kWh."""
        base = 0.42 - 0.13 * pv_norm                      # solar depresses it
        evening = 0.10 * np.exp(-0.5 * ((self.hour - 19) / 2.0) ** 2)
        night = -0.04 * np.exp(-0.5 * ((self.hour - 3) / 2.5) ** 2)
        return np.clip(base + evening + night + 0.03 * self._ar1(0.8, 1.0),
                       0.06, None)

    def _make_tariff(self):
        p = np.full(self.T, 0.11)
        p[(self.hour >= 8) & (self.hour < 12)] = 0.17
        p[(self.hour >= 16) & (self.hour < 21)] = 0.28
        p[(self.hour >= 0) & (self.hour < 6)] = 0.075
        weekend = (self.day % 7) >= 5
        p[weekend] = np.minimum(p[weekend], 0.13)
        return p

    def _make_base_load(self):
        occ = self.occupancy
        return self.b.base_load_kw * (0.34 + 0.66 * occ) * (
            1.0 + 0.05 * self._ar1(0.75, 1.0))

    def _make_occupancy(self):
        prof = np.clip(
            np.exp(-0.5 * ((self.hour - 13.0) / 4.1) ** 2), 0, 1)
        prof = np.where((self.hour < 7) | (self.hour > 20), 0.05, prof)
        weekend = (self.day % 7) >= 5
        prof = np.where(weekend, prof * 0.12, prof)
        return prof

    def _make_fleet(self):
        """Per-day, per-cluster mobility realisation."""
        f, D = self.f, self.s.days
        recs = []
        for c, (a_mu, d_mu, soc_mu, soc_req, p_part) in enumerate(f.profiles):
            n = f.cluster_sizes[c]
            arr = self.rng.normal(a_mu, f.arrive_sd, D)
            dep_decl = self.rng.normal(d_mu, f.depart_sd, D)
            dep_decl = np.maximum(dep_decl, arr + 2.0)
            early = self.rng.random(D) < f.early_hazard
            shift = self.rng.exponential(f.early_shift_mu, D) * early
            dep_act = np.maximum(dep_decl - shift, arr + 1.0)
            soc_arr = np.clip(self.rng.normal(soc_mu, 0.11, D), 0.12, 0.95)
            part = self.rng.random(D) < p_part
            recs.append(dict(cluster=c, n=n, arr=arr, dep_decl=dep_decl,
                             dep_act=dep_act, soc_arr=soc_arr,
                             soc_req=np.full(D, soc_req), part=part))
        return recs

    # -- helpers ---------------------------------------------------------------
    def _ar1(self, phi, sd):
        e = self.rng.normal(0, sd, self.T)
        x = np.zeros(self.T)
        for t in range(1, self.T):
            x[t] = phi * x[t - 1] + e[t]
        return x

    def _build(self):
        self.occupancy = self._make_occupancy()
        self.tamb = self._make_ambient()
        self.solar = self._make_solar()
        self.pv = self.b.pv_kwp * self.solar
        self.carbon = self._make_carbon(self.solar / (self.solar.max() + 1e-9))
        self.price = self._make_tariff()
        self.base_load = self._make_base_load()
        self.fleet = self._make_fleet()
        self.conn, self.conn_decl, self.ev_soc0, self.ev_req, self.ev_part = \
            self._expand_fleet()
        self.events = self._make_events()

    def _expand_fleet(self):
        """Hourly connection matrices, [n_clusters, T]."""
        K, T = self.f.n_clusters, self.T
        conn = np.zeros((K, T))
        conn_d = np.zeros((K, T))
        soc0 = np.zeros((K, self.s.days))
        req = np.zeros((K, self.s.days))
        part = np.zeros((K, self.s.days))
        for r in self.fleet:
            c = r["cluster"]
            for d in range(self.s.days):
                h = np.arange(24)
                on_a = (h >= r["arr"][d]) & (h < r["dep_act"][d])
                on_d = (h >= r["arr"][d]) & (h < r["dep_decl"][d])
                conn[c, d * 24:(d + 1) * 24] = on_a * r["n"]
                conn_d[c, d * 24:(d + 1) * 24] = on_d * r["n"]
                soc0[c, d] = r["soc_arr"][d]
                req[c, d] = r["soc_req"][d]
                part[c, d] = float(r["part"][d])
        return conn, conn_d, soc0, req, part

    def _make_events(self):
        """Grid service requests: (t_start, duration, magnitude_kw).

        Magnitudes are expressed as a fraction of the *expected* import at
        the requested hour, which is how a system operator would size a
        request; an absolute magnitude would frequently exceed the site's
        entire demand and make the request trivially infeasible.
        """
        nominal = np.maximum(
            self.base_load + 0.40 * self.b.hvac_max - self.pv, 15.0)
        ev = []
        for d in range(self.s.days):
            k = self.rng.poisson(self.s.n_events_per_day)
            for _ in range(int(k)):
                h = int(np.clip(self.rng.normal(18.0, 2.6), 7, 21))
                dur = int(self.rng.choice([1, 2, 3], p=[0.45, 0.4, 0.15]))
                t0 = d * 24 + h
                if t0 + dur >= self.T:
                    continue
                frac = float(np.clip(self.rng.normal(0.42, 0.15), 0.12, 0.80))
                mag = float(frac * nominal[t0:t0 + dur].mean())
                ev.append((t0, dur, mag))
        return sorted(ev)

    def resize_events(self, ref_schedule, frac=(0.15, 0.60), min_import=40.0):
        """Re-issue grid requests against a declared day-ahead schedule.

        A flexibility market asks for a reduction relative to the position a
        site has *scheduled*, not relative to its unmanaged demand. Sizing
        requests against an unmanaged profile makes them unsatisfiable for an
        already-optimised building, which measures nothing useful.
        """
        rng = np.random.default_rng(self.s.seed + 4242)
        ok = np.where(ref_schedule >= min_import)[0]
        ok = ok[(ok % 24 >= 7) & (ok % 24 <= 21)]
        n_target = int(self.s.n_events_per_day * self.s.days)
        ev = []
        if len(ok) == 0:
            self.events = []
            return self.events
        for t0 in rng.choice(ok, size=min(n_target, len(ok)), replace=False):
            dur = int(rng.choice([1, 2, 3], p=[0.45, 0.40, 0.15]))
            t0 = int(t0)
            if t0 + dur >= self.T:
                continue
            win = ref_schedule[t0:t0 + dur]
            if win.mean() < min_import:
                continue
            fr = float(np.clip(rng.normal(np.mean(frac), 0.13),
                               frac[0], frac[1]))
            ev.append((t0, dur, float(fr * win.mean())))
        self.events = sorted(ev)
        return self.events

    # -- forecasts -------------------------------------------------------------
    def forecast(self, t, H, severity=None, perfect=False):
        """Return forecast dict over [t, t+H)."""
        sev = self.s.forecast_severity if severity is None else severity
        sl = slice(t, t + H)
        out = dict(tamb=self.tamb[sl].copy(), pv=self.pv[sl].copy(),
                   load=self.base_load[sl].copy(),
                   carbon=self.carbon[sl].copy(),
                   price=self.price[sl].copy(),
                   conn=self.conn[:, sl].copy())
        if perfect or sev == 0:
            return out
        rng = np.random.default_rng(self.s.seed + 7919 * t)
        n = out["tamb"].shape[0]
        ramp = np.sqrt(np.arange(1, n + 1))          # error grows with horizon
        out["tamb"] += rng.normal(0, 0.55 * sev, n) * ramp
        out["pv"] = np.clip(
            out["pv"] * (1 + rng.normal(0, 0.13 * sev, n) * ramp), 0, None)
        out["load"] *= (1 + rng.normal(0, 0.055 * sev, n) * ramp)
        out["carbon"] = np.clip(
            out["carbon"] * (1 + rng.normal(0, 0.075 * sev, n) * ramp),
            0.05, None)
        # EV availability forecast uses DECLARED departure -> optimistic
        out["conn"] = self.conn_decl[:, sl].copy()
        return out


# ----------------------------------------------------------------------------
# 3. Plant simulator
# ----------------------------------------------------------------------------


class Plant:
    """Physical state evolution. One step at a time, driven by a controller."""

    def __init__(self, sc: Scenario):
        self.sc = sc
        self.b, self.f, self.s = sc.b, sc.f, sc.s
        self.reset()

    def reset(self):
        self.t = 0
        self.theta = self.b.theta_set
        self.soc = self.b.bat_soc0
        self.ev_soc = np.array(
            [self.sc.ev_soc0[c, 0] for c in range(self.f.n_clusters)])
        self.prev_conn = np.zeros(self.f.n_clusters)
        self.log = []

    # ---- physics -------------------------------------------------------------
    def thermal_step(self, theta, tamb, occ, sol, p_hvac):
        b = self.b
        cop = b.cop_heat if b.winter_peaking else b.cop_cool
        sign = +1.0 if b.winter_peaking else -1.0
        q_gain = (b.gain_occ_kw * occ + b.sol_gain_frac * b.pv_kwp * sol) * 1e3
        q_env = (tamb - theta) / b.R
        q_hvac = sign * p_hvac * 1e3 * cop
        return theta + (self.s.dt * 3600.0 / b.C) * (q_env + q_gain + q_hvac)

    def ev_arrivals(self, t):
        """Reset cluster SOC on arrival, return departures needing settlement."""
        d = t // 24
        conn = self.sc.conn[:, t]
        arrived = (conn > 0) & (self.prev_conn == 0)
        departed = (conn == 0) & (self.prev_conn > 0)
        short = np.zeros(self.f.n_clusters)
        for c in range(self.f.n_clusters):
            if arrived[c]:
                self.ev_soc[c] = self.sc.ev_soc0[c, min(d, self.s.days - 1)]
            if departed[c]:
                req = self.sc.ev_req[c, min(d, self.s.days - 1)]
                short[c] = max(0.0, req - self.ev_soc[c]) * \
                    self.f.batt_kwh * self.prev_conn[c]
        self.prev_conn = conn.copy()
        return short

    def step(self, p_hvac, p_bat, p_ev):
        """p_bat > 0 discharge; p_ev > 0 discharge (V2B), per cluster."""
        t, sc, b, f = self.t, self.sc, self.b, self.f
        short = self.ev_arrivals(t)

        # --- battery
        p_bat = float(np.clip(p_bat, -b.bat_kw, b.bat_kw))
        if p_bat >= 0:
            e = p_bat * self.s.dt / b.bat_eff
        else:
            e = p_bat * self.s.dt * b.bat_eff
        new_soc = self.soc - e / b.bat_kwh
        if not (0.1 <= new_soc <= 0.95):
            new_soc = float(np.clip(new_soc, 0.1, 0.95))
            e = (self.soc - new_soc) * b.bat_kwh
            p_bat = e / self.s.dt * (b.bat_eff if e < 0 else 1 / b.bat_eff)
        bat_throughput = abs(e)
        self.soc = new_soc

        # --- EV clusters
        conn = sc.conn[:, t]
        p_ev = np.asarray(p_ev, float).copy()
        ev_throughput = 0.0
        for c in range(f.n_clusters):
            if conn[c] <= 0:
                p_ev[c] = 0.0
                continue
            cap = conn[c] * f.charger_kw
            p_ev[c] = float(np.clip(p_ev[c], -cap, cap))
            e_c = p_ev[c] * self.s.dt / (f.v2b_eff if p_ev[c] >= 0 else 1.0)
            pool = conn[c] * f.batt_kwh
            ns = self.ev_soc[c] - e_c / pool
            if not (0.15 <= ns <= 0.98):
                ns = float(np.clip(ns, 0.15, 0.98))
                e_c = (self.ev_soc[c] - ns) * pool
                p_ev[c] = e_c / self.s.dt
            ev_throughput += abs(e_c)
            self.ev_soc[c] = ns

        # --- thermal
        p_hvac = float(np.clip(p_hvac, 0, b.hvac_max))
        self.theta = self.thermal_step(
            self.theta, sc.tamb[t], sc.occupancy[t], sc.solar[t], p_hvac)

        # --- electrical balance
        net = sc.base_load[t] + p_hvac - sc.pv[t] - p_bat - p_ev.sum()
        imp = max(net, 0.0)
        exp = max(-net, 0.0)

        rec = dict(
            t=t, theta=self.theta, p_hvac=p_hvac, p_bat=p_bat,
            p_ev=p_ev.sum(), soc=self.soc, net=net, imp=imp, exp=exp,
            co2=imp * sc.carbon[t] * self.s.dt,
            cost=(imp * sc.price[t]
                  - exp * sc.price[t] * self.s.export_price_frac) * self.s.dt
            + bat_throughput * b.bat_deg_cost + ev_throughput * f.deg_cost,
            deg=bat_throughput * b.bat_deg_cost + ev_throughput * f.deg_cost,
            cyc_bat=bat_throughput / b.bat_kwh,
            discomfort=max(0.0, b.theta_lo - self.theta)
            + max(0.0, self.theta - b.theta_hi),
            ev_short=short.sum(),
        )
        self.log.append(rec)
        self.t += 1
        return rec


# ----------------------------------------------------------------------------
# 4. Optimisation core (compiled once, re-solved with parameters)
# ----------------------------------------------------------------------------


class MPCCore:
    """Linear MPC over the horizon. Compiled once via DPP for fast re-solve."""

    def __init__(self, b: BuildingCfg, f: FleetCfg, s: SimCfg):
        self.b, self.f, self.s = b, f, s
        H, K = s.horizon, f.n_clusters
        self.H, self.K = H, K

        # variables
        ph = cp.Variable(H, nonneg=True)
        pb = cp.Variable(H)
        pe = cp.Variable((K, H))
        th = cp.Variable(H + 1)
        sb = cp.Variable(H + 1)
        se = cp.Variable((K, H + 1))
        imp = cp.Variable(H, nonneg=True)
        exp_ = cp.Variable(H, nonneg=True)
        sl_lo = cp.Variable(H, nonneg=True)
        sl_hi = cp.Variable(H, nonneg=True)
        sl_ev = cp.Variable(K, nonneg=True)
        sl_srv = cp.Variable(H, nonneg=True)
        abs_b = cp.Variable(H, nonneg=True)
        abs_e = cp.Variable((K, H), nonneg=True)

        # parameters
        P = {}
        P["tamb"] = cp.Parameter(H)
        P["gain"] = cp.Parameter(H)
        P["pv"] = cp.Parameter(H)
        P["load"] = cp.Parameter(H)
        P["carbon"] = cp.Parameter(H, nonneg=True)
        P["price"] = cp.Parameter(H, nonneg=True)
        P["cap"] = cp.Parameter((K, H), nonneg=True)     # charger capacity
        P["invpool"] = cp.Parameter((K, H), nonneg=True)  # dt / energy pool
        P["th0"] = cp.Parameter()
        P["sb0"] = cp.Parameter()
        P["se0"] = cp.Parameter(K)
        P["soctgt"] = cp.Parameter(K, nonneg=True)   # required SOC at departure
        P["depmask"] = cp.Parameter((K, H + 1), nonneg=True)  # 1 at departure
        P["wpeak"] = cp.Parameter(nonneg=True)
        P["srvcap"] = cp.Parameter(H)   # import ceiling implied by commitment
        self.P = P

        cons = [th[0] == P["th0"], sb[0] == P["sb0"], se[:, 0] == P["se0"]]

        cop = b.cop_heat if b.winter_peaking else b.cop_cool
        sign = 1.0 if b.winter_peaking else -1.0
        k = s.dt * 3600.0 / b.C
        for t in range(H):
            cons += [
                th[t + 1] == th[t] + k * (
                    (P["tamb"][t] - th[t]) / b.R + P["gain"][t]
                    + sign * ph[t] * 1e3 * cop),
                sb[t + 1] == sb[t] - pb[t] * s.dt / b.bat_kwh / b.bat_eff,
                ph[t] <= b.hvac_max,
                pb[t] <= b.bat_kw, pb[t] >= -b.bat_kw,
                abs_b[t] >= pb[t], abs_b[t] >= -pb[t],
                th[t + 1] >= b.theta_lo - sl_lo[t],
                th[t + 1] <= b.theta_hi + sl_hi[t],
                sb[t + 1] >= 0.10, sb[t + 1] <= 0.95,
                imp[t] - exp_[t] == P["load"][t] + ph[t] - P["pv"][t]
                - pb[t] - cp.sum(pe[:, t]),
                # committed grid service: soft import ceiling
                imp[t] <= P["srvcap"][t] + sl_srv[t],
            ]
            for c in range(K):
                cons += [
                    pe[c, t] <= P["cap"][c, t], pe[c, t] >= -P["cap"][c, t],
                    abs_e[c, t] >= pe[c, t], abs_e[c, t] >= -pe[c, t],
                    se[c, t + 1] == se[c, t]
                    - cp.multiply(P["invpool"][c, t], pe[c, t]),
                    se[c, t + 1] >= 0.15, se[c, t + 1] <= 0.98,
                ]
        # the departure requirement must bind at THIS departure, not at the
        # end of the horizon -- otherwise the controller discharges a vehicle
        # today and satisfies the constraint by recharging it tomorrow.
        for c in range(K):
            cons += [cp.sum(cp.multiply(P["depmask"][c, :], se[c, :]))
                     >= P["soctgt"][c] - sl_ev[c]]

        obj = (
            cp.sum(cp.multiply(P["carbon"], imp)) * 1.0
            + cp.sum(cp.multiply(P["price"], imp)) * 2.0
            - cp.sum(cp.multiply(P["price"], exp_)) * 2.0 * s.export_price_frac
            + P["wpeak"] * cp.max(imp) * 0.02
            + 22.0 * cp.sum(sl_lo + sl_hi)
            + 3000.0 * cp.sum(sl_ev)
            + 40.0 * cp.sum(sl_srv)
            + b.bat_deg_cost * cp.sum(abs_b)
            + f.deg_cost * cp.sum(abs_e)
        )
        self.prob = cp.Problem(cp.Minimize(obj), cons)
        self.vars = dict(ph=ph, pb=pb, pe=pe, th=th, sb=sb, se=se,
                         imp=imp, sl_lo=sl_lo, sl_hi=sl_hi)

    def build_envelope_problem(self):
        """Auxiliary problem: maximise the mean import reduction achievable
        over a window, subject to every physical, comfort and mobility
        constraint. This is the physical flexibility envelope; the conformal
        layer then only has to correct for forecast and behavioural error,
        not for the controller's own optimisation trade-offs."""
        b, f, s = self.b, self.f, self.s
        H, K = self.H, self.K
        ph = cp.Variable(H, nonneg=True)
        pb = cp.Variable(H)
        pe = cp.Variable((K, H))
        th = cp.Variable(H + 1)
        sb = cp.Variable(H + 1)
        se = cp.Variable((K, H + 1))
        imp = cp.Variable(H, nonneg=True)
        exp_ = cp.Variable(H, nonneg=True)
        Q = {}
        for k in ["tamb", "gain", "pv", "load"]:
            Q[k] = cp.Parameter(H)
        Q["cap"] = cp.Parameter((K, H), nonneg=True)
        Q["invpool"] = cp.Parameter((K, H), nonneg=True)
        Q["win"] = cp.Parameter(H, nonneg=True)      # 1 inside the event
        Q["th0"] = cp.Parameter()
        Q["sb0"] = cp.Parameter()
        Q["se0"] = cp.Parameter(K)
        Q["soctgt"] = cp.Parameter(K, nonneg=True)
        Q["depmask"] = cp.Parameter((K, H + 1), nonneg=True)
        cons = [th[0] == Q["th0"], sb[0] == Q["sb0"], se[:, 0] == Q["se0"]]
        cop = b.cop_heat if b.winter_peaking else b.cop_cool
        sign = 1.0 if b.winter_peaking else -1.0
        kk = s.dt * 3600.0 / b.C
        for t in range(H):
            cons += [
                th[t + 1] == th[t] + kk * ((Q["tamb"][t] - th[t]) / b.R
                                           + Q["gain"][t]
                                           + sign * ph[t] * 1e3 * cop),
                sb[t + 1] == sb[t] - pb[t] * s.dt / b.bat_kwh / b.bat_eff,
                ph[t] <= b.hvac_max,
                pb[t] <= b.bat_kw, pb[t] >= -b.bat_kw,
                th[t + 1] >= b.theta_lo, th[t + 1] <= b.theta_hi,
                sb[t + 1] >= 0.10, sb[t + 1] <= 0.95,
                imp[t] - exp_[t] == Q["load"][t] + ph[t] - Q["pv"][t]
                - pb[t] - cp.sum(pe[:, t]),
            ]
            for c in range(K):
                cons += [
                    pe[c, t] <= Q["cap"][c, t], pe[c, t] >= -Q["cap"][c, t],
                    se[c, t + 1] == se[c, t]
                    - cp.multiply(Q["invpool"][c, t], pe[c, t]),
                    se[c, t + 1] >= 0.15, se[c, t + 1] <= 0.98,
                ]
        for c in range(K):
            # mobility is inviolable when computing what can be PROMISED
            cons += [cp.sum(cp.multiply(Q["depmask"][c, :], se[c, :]))
                     >= Q["soctgt"][c]]
        self.env_prob = cp.Problem(
            cp.Minimize(cp.sum(cp.multiply(Q["win"], imp))), cons)
        self.Q, self.env_imp = Q, imp

    def envelope(self, fc, state, soctgt, depmask, win, base):
        """Return the certified-envelope point estimate phi_hat (kW)."""
        if not hasattr(self, "env_prob"):
            self.build_envelope_problem()
        Q = self.Q
        occ = fc["occ"]
        Q["tamb"].value = fc["tamb"]
        Q["gain"].value = (self.b.gain_occ_kw * occ
                           + self.b.sol_gain_frac * self.b.pv_kwp
                           * fc["sol"]) * 1e3
        Q["pv"].value = fc["pv"]
        Q["load"].value = fc["load"]
        Q["cap"].value = fc["conn"] * self.f.charger_kw
        Q["invpool"].value = self.s.dt / np.maximum(
            fc["conn"] * self.f.batt_kwh, 1.0)
        Q["win"].value = win
        Q["th0"].value = state["theta"]
        Q["sb0"].value = state["soc"]
        Q["se0"].value = state["ev_soc"]
        Q["soctgt"].value = np.clip(soctgt, 0.0, 0.98)
        Q["depmask"].value = depmask
        try:
            self.env_prob.solve(solver=cp.CLARABEL, warm_start=True)
        except Exception:
            return None
        if self.env_imp.value is None:
            return None
        n = max(win.sum(), 1.0)
        return float(max(0.0, (base * win).sum() / n
                         - (self.env_imp.value * win).sum() / n))

    def solve(self, fc, state, srvcap, soctgt, depmask, wpeak=1.0):
        P, H, K = self.P, self.H, self.K
        occ = fc["occ"]
        P["tamb"].value = fc["tamb"]
        P["gain"].value = (self.b.gain_occ_kw * occ
                           + self.b.sol_gain_frac * self.b.pv_kwp
                           * fc["sol"]) * 1e3
        P["pv"].value = fc["pv"]
        P["load"].value = fc["load"]
        P["carbon"].value = np.maximum(fc["carbon"], 1e-3)
        P["price"].value = np.maximum(fc["price"], 1e-3)
        P["cap"].value = fc["conn"] * self.f.charger_kw
        P["invpool"].value = self.s.dt / np.maximum(
            fc["conn"] * self.f.batt_kwh, 1.0)
        P["th0"].value = state["theta"]
        P["sb0"].value = state["soc"]
        P["se0"].value = state["ev_soc"]
        P["soctgt"].value = np.clip(soctgt, 0.0, 0.98)
        P["depmask"].value = depmask
        P["wpeak"].value = wpeak
        P["srvcap"].value = srvcap
        try:
            self.prob.solve(solver=cp.CLARABEL, warm_start=True)
            if self.vars["ph"].value is None:
                raise RuntimeError
        except Exception:
            try:
                self.prob.solve(solver=cp.SCS, warm_start=True)
            except Exception:
                return None
        if self.vars["ph"].value is None:
            return None
        return {k: (v.value.copy() if v.value is not None else None)
                for k, v in self.vars.items()}


# ----------------------------------------------------------------------------
# 5. Conformal certification layer
# ----------------------------------------------------------------------------


class ConformalCertifier:
    """Split-conformal certification of deliverable flexibility.

    Score  s = phi_hat - phi_realised   (positive = over-estimate).
    Certified quantity  P_down = phi_hat - q_(1-alpha).

    Variants
    --------
    mode='none'      : no certification, commit the point estimate
    mode='split'     : plain split conformal, one global calibration set
    mode='mondrian'  : conditioned on operating regime bins
    mode='adaptive'  : Mondrian + online alpha adaptation (ACI)
    """

    def __init__(self, alpha=0.10, mode="adaptive", min_n=30,
                 lr=0.02, halflife=400.0):
        self.alpha0 = alpha
        self.mode = mode
        self.min_n = min_n
        self.lr = lr
        self.halflife = halflife
        self.store = {}          # bin -> list of (score, age_index)
        self.alpha_t = {}        # bin -> current effective alpha
        self.k = 0

    # -- regime binning --------------------------------------------------------
    @staticmethod
    def bin_of(ctx):
        """Operating-regime bin.

        Deliverability is governed by which resources actually have headroom,
        so the regime is defined by fleet connection, thermal headroom and
        stationary state of charge -- not by the calendar.
        """
        if ctx is None:
            return "all"
        c = ctx.get("conn_frac", 0.0)
        th = ctx.get("th_head", 0.0)
        sb = ctx.get("soc", 0.5)
        return "{}{}{}".format("H" if c > 0.45 else "L",
                               "H" if th > 0.9 else "L",
                               "H" if sb > 0.45 else "L")

    def _key(self, ctx):
        if self.mode in ("none", "split"):
            return "all"
        return self.bin_of(ctx)

    def quantile(self, ctx):
        """Return the conformal correction q for the current context."""
        if self.mode == "none":
            return 0.0
        key = self._key(ctx)
        rec = self.store.get(key, [])
        if len(rec) < self.min_n:
            # fall back to the pooled set while the bin is cold
            rec = [r for v in self.store.values() for r in v]
            if len(rec) < self.min_n:
                return None                      # not yet certifiable
        s = np.array([r[0] for r in rec])
        w = np.exp(-np.log(2) * (self.k - np.array([r[1] for r in rec]))
                   / self.halflife) if self.mode == "adaptive" else None
        a = self.alpha_t.get(key, self.alpha0)
        a = float(np.clip(a, 0.005, 0.5))
        if w is None:
            return float(np.quantile(s, 1 - a, method="higher"))
        idx = np.argsort(s)
        s_, w_ = s[idx], w[idx]
        cw = np.cumsum(w_) / np.sum(w_)
        j = int(np.searchsorted(cw, 1 - a))
        return float(s_[min(j, len(s_) - 1)])

    def certify(self, phi_hat, ctx=None):
        q = self.quantile(ctx)
        if q is None:
            return None                          # abstain: cannot certify yet
        return max(0.0, phi_hat - q)

    def update(self, phi_hat, phi_real, ctx=None, covered=None):
        key = self._key(ctx)
        self.k += 1
        self.store.setdefault(key, []).append((phi_hat - phi_real, self.k))
        if len(self.store[key]) > 3000:
            self.store[key] = self.store[key][-3000:]
        if self.mode == "adaptive" and covered is not None:
            a = self.alpha_t.get(key, self.alpha0)
            # Adaptive Conformal Inference (Gibbs & Candes)
            self.alpha_t[key] = a + self.lr * (self.alpha0 - (0.0 if covered
                                                              else 1.0))


# ----------------------------------------------------------------------------
# 6. Controllers
# ----------------------------------------------------------------------------


class BaseController:
    name = "base"
    uses_mpc = False
    ref_import = None
    respond = True          # False -> ignore grid requests (counterfactual run)

    def note_naive_commitment(self, t):
        """Rule-based controllers accept every request in full: they have no
        mechanism to evaluate whether they can honour it."""
        ev = self.event_starting(t) if self.respond else None
        if ev is not None:
            self.commitments.append(dict(t0=ev[0], dur=ev[1], requested=ev[2],
                                         promised=ev[2], decision="act"))

    def __init__(self, sc: Scenario, core: MPCCore | None = None):
        self.sc, self.core = sc, core
        self.b, self.f, self.s = sc.b, sc.f, sc.s
        self.commitments = []     # (t0, dur, requested, promised, decision)

    # -- helpers ---------------------------------------------------------------
    def horizon_fc(self, t, perfect=False):
        H = self.s.horizon
        H = min(H, self.sc.T - t)
        fc = self.sc.forecast(t, H, perfect=perfect)
        fc["occ"] = self.sc.occupancy[t:t + H]
        fc["sol"] = self.sc.solar[t:t + H]
        if H < self.s.horizon:                    # pad tail
            pad = self.s.horizon - H
            for k in fc:
                a = fc[k]
                fc[k] = (np.pad(a, ((0, 0), (0, pad)), mode="edge")
                         if a.ndim == 2 else np.pad(a, (0, pad), mode="edge"))
        return fc

    def soctgt(self, t, plant, fc):
        """Required departure SOC and the horizon index where it binds.

        The mask marks the first forecast departure of each cluster inside
        the horizon. Clusters not currently connected impose no requirement.
        """
        d = min(t // 24, self.s.days - 1)
        K, H = self.f.n_clusters, self.s.horizon
        tgt = np.zeros(K)
        mask = np.zeros((K, H + 1))
        conn = fc["conn"]
        for c in range(K):
            if conn[c, 0] <= 0:
                continue
            tgt[c] = self.sc.ev_req[c, d]
            k_dep = H                       # default: end of horizon
            for k in range(1, H):
                if conn[c, k] <= 0:
                    k_dep = k
                    break
            mask[c, k_dep] = 1.0
        return tgt, mask

    def active_event(self, t):
        for (t0, dur, mag) in self.sc.events:
            if t0 <= t < t0 + dur:
                return (t0, dur, mag)
        return None

    def event_starting(self, t):
        for (t0, dur, mag) in self.sc.events:
            if t0 == t:
                return (t0, dur, mag)
        return None

    def act(self, t, plant):
        raise NotImplementedError


class Uncontrolled(BaseController):
    """B1: thermostat + immediate EV charging, no storage participation."""
    name = "B1 uncontrolled"

    def act(self, t, plant):
        self.note_naive_commitment(t)
        b = self.b
        err = plant.theta - b.theta_set
        gain = b.hvac_max / 1.6
        ph = np.clip((-err if b.winter_peaking else err) * gain, 0, b.hvac_max)
        pe = -self.sc.conn[:, t] * self.f.charger_kw      # charge at full rate
        return ph, 0.0, pe


class TOURule(BaseController):
    """B2: time-of-use rule."""
    name = "B2 TOU rule"

    def act(self, t, plant):
        self.note_naive_commitment(t)
        b, sc = self.b, self.sc
        err = plant.theta - b.theta_set
        gain = b.hvac_max / 1.6
        ph = np.clip((-err if b.winter_peaking else err) * gain, 0, b.hvac_max)
        cheap = sc.price[t] <= 0.12
        peak = sc.price[t] >= 0.25
        pb = -b.bat_kw * 0.8 if cheap else (b.bat_kw * 0.8 if peak else 0.0)
        pe = np.zeros(self.f.n_clusters)
        for c in range(self.f.n_clusters):
            if sc.conn[c, t] > 0:
                cap = sc.conn[c, t] * self.f.charger_kw
                pe[c] = -cap if cheap else (0.0 if peak else -0.45 * cap)
        return ph, pb, pe


class CarbonRule(BaseController):
    """B3: carbon-responsive rule (after Wang et al. 2023)."""
    name = "B3 carbon rule"

    def act(self, t, plant):
        self.note_naive_commitment(t)
        b, sc = self.b, self.sc
        lo = np.quantile(sc.carbon, 0.35)
        hi = np.quantile(sc.carbon, 0.70)
        err = plant.theta - b.theta_set
        gain = b.hvac_max / 1.6
        bias = -0.7 if sc.carbon[t] < lo else (0.7 if sc.carbon[t] > hi else 0)
        tgt = b.theta_set + (bias if not b.winter_peaking else -bias)
        err = plant.theta - tgt
        ph = np.clip((-err if b.winter_peaking else err) * gain, 0, b.hvac_max)
        pb = (-b.bat_kw * 0.85 if sc.carbon[t] < lo
              else (b.bat_kw * 0.85 if sc.carbon[t] > hi else 0.0))
        pe = np.zeros(self.f.n_clusters)
        d = min(t // 24, self.s.days - 1)
        for c in range(self.f.n_clusters):
            if sc.conn[c, t] > 0:
                cap = sc.conn[c, t] * self.f.charger_kw
                urgent = plant.ev_soc[c] < sc.ev_req[c, d] - 0.12
                if urgent or sc.carbon[t] < lo:
                    pe[c] = -cap
                elif sc.carbon[t] > hi and sc.ev_part[c, d] > 0:
                    pe[c] = 0.55 * cap
        return ph, pb, pe


class MPCController(BaseController):
    """B4/B5/B6: predictive control. Commits the full request if the model
    says it is feasible -- the 'happy path' assumption."""
    uses_mpc = True

    def __init__(self, sc, core, perfect=False, robust=0.0, name=None):
        super().__init__(sc, core)
        self.perfect = perfect
        self.robust = robust           # conservatism margin, fraction
        self.name = name or ("B4 MPC perfect" if perfect else "B5 MPC real")
        self._plan = None

    def baseline_import(self, t, fc):
        """Reference import against which a service reduction is measured."""
        H = self.s.horizon
        if getattr(self, "ref_import", None) is not None:
            r = self.ref_import[t:t + H]
            if len(r) < H:
                r = np.pad(r, (0, H - len(r)), mode="edge")
            return r
        return np.maximum(fc["load"] - fc["pv"] + 0.35 * self.b.hvac_max, 0)

    def decide_commitment(self, t, ev, fc, plant):
        """Return promised reduction (kW) for an event starting now."""
        t0, dur, mag = ev
        return mag                                   # commit in full

    def act(self, t, plant):
        fc = self.horizon_fc(t, perfect=self.perfect)
        if self.robust > 0:
            fc = dict(fc)
            fc["pv"] = fc["pv"] * (1 - self.robust)
            fc["load"] = fc["load"] * (1 + self.robust)
            fc["conn"] = fc["conn"] * (1 - self.robust)
        H = self.s.horizon
        base = self.baseline_import(t, fc)

        ev = self.event_starting(t) if self.respond else None
        if ev is not None:
            promised = self.decide_commitment(t, ev, fc, plant)
            self.commitments.append(
                dict(t0=ev[0], dur=ev[1], requested=ev[2],
                     promised=promised, decision=self._decision_label(
                         promised, ev[2])))

        srvcap = np.full(H, 1e5)
        if self.respond:
            for (tt0, dd, mm) in self.sc.events:
                for k in range(H):
                    if tt0 <= t + k < tt0 + dd:
                        pr = self._promised_for(tt0)
                        if pr is not None and pr > 0:
                            srvcap[k] = max(base[k] - pr, 0.0)
        tgt, mask = self.soctgt(t, plant, fc)
        st = dict(theta=plant.theta, soc=plant.soc,
                  ev_soc=plant.ev_soc.copy())
        sol = self.core.solve(fc, st, srvcap, tgt, mask)
        if sol is None:                              # genuine infeasibility
            sol = self.core.solve(fc, st, np.full(H, 1e5), tgt, mask)
        if sol is None:
            return 0.35 * self.b.hvac_max, 0.0, np.zeros(self.f.n_clusters)
        return sol["ph"][0], sol["pb"][0], sol["pe"][:, 0]

    def _decision_label(self, promised, requested):
        if promised <= 1e-6:
            return "abstain"
        return "act" if promised >= requested - 1e-6 else "derate"

    def _promised_for(self, t0):
        for c in self.commitments:
            if c["t0"] == t0:
                return c["promised"]
        return None


class CFOController(MPCController):
    """Proposed: certified envelope + selective commitment policy."""

    def __init__(self, sc, core, certifier: ConformalCertifier,
                 use_risk=True, use_transfer=True, use_info=True,
                 p_explore=0.10, n_commission=30, name="P CFO"):
        super().__init__(sc, core, perfect=False, robust=0.0, name=name)
        self.cert = certifier
        self.use_risk = use_risk
        self.use_transfer = use_transfer
        self.use_info = use_info          # information term -> exploration
        self.p_explore = p_explore if use_info else 0.0
        self.n_commission = n_commission   # full-commitment commissioning
        self.envelopes = []
        self._rng = np.random.default_rng(sc.s.seed + 101)

    def phi_hat_lp(self, t, dur, fc, plant, base):
        """Physical envelope from the auxiliary LP (preferred estimator)."""
        H = self.s.horizon
        win = np.zeros(H)
        win[:min(dur, H)] = 1.0
        tgt, mask = self.soctgt(t, plant, fc)
        st = dict(theta=plant.theta, soc=plant.soc,
                  ev_soc=plant.ev_soc.copy())
        v = self.core.envelope(fc, st, tgt, mask, win, base)
        return v if v is not None else self.phi_hat(t, dur, fc, plant)

    # -- heuristic fallback ---------------------------------------------------
    def phi_hat(self, t, dur, fc, plant):
        """Point estimate of sustainable import reduction over `dur` hours."""
        b, f, sc = self.b, self.f, self.sc
        d = min(t // 24, self.s.days - 1)
        # battery contribution limited by energy and power
        e_bat = max(0.0, (plant.soc - 0.10)) * b.bat_kwh
        p_bat = min(b.bat_kw, e_bat / max(dur, 1e-6))
        # EV contribution: only participating, only surplus above requirement
        p_ev = 0.0
        for c in range(f.n_clusters):
            n = fc["conn"][c, 0]
            if n <= 0 or sc.ev_part[c, d] <= 0:
                continue
            surplus = max(0.0, plant.ev_soc[c] - sc.ev_req[c, d]) \
                * f.batt_kwh * n
            p_ev += min(n * f.charger_kw, surplus / max(dur, 1e-6))
        # HVAC: coasting within the comfort band
        head = (b.theta_hi - plant.theta) if not b.winter_peaking \
            else (plant.theta - b.theta_lo)
        cop = b.cop_heat if b.winter_peaking else b.cop_cool
        e_th = max(0.0, head) * b.C / 3.6e6 / cop        # kWh of avoided input
        p_th = min(b.hvac_max * 0.75, e_th / max(dur, 1e-6))
        return p_bat + p_ev + p_th

    def context(self, t, plant=None):
        conn = self.sc.conn[:, t].sum()
        cmax = sum(self.f.cluster_sizes)
        b = self.b
        if plant is None:
            return dict(conn_frac=conn / max(cmax, 1), th_head=0.0, soc=0.5)
        head = (b.theta_hi - plant.theta) if not b.winter_peaking \
            else (plant.theta - b.theta_lo)
        return dict(conn_frac=conn / max(cmax, 1),
                    th_head=float(max(head, 0.0)), soc=float(plant.soc))

    def decide_commitment(self, t, ev, fc, plant):
        t0, dur, mag = ev
        ctx = self.context(t, plant)
        base = self.baseline_import(t, fc)
        ph_hat = self.phi_hat_lp(t, dur, fc, plant, base)
        certified = self.cert.certify(ph_hat, ctx)
        # Information term: a de-rated promise censors its own calibration
        # data (we never learn what the system COULD have delivered).
        # Occasionally commit in full to generate an uncensored observation.
        n_seen = len(self.envelopes)
        explore = (n_seen < self.n_commission
                   or bool(self._rng.random() < self.p_explore))
        self.envelopes.append(dict(t=t, dur=dur, phi_hat=ph_hat,
                                   certified=certified, requested=mag,
                                   explore=explore, **ctx))
        if explore:
            return mag
        if certified is None:
            # cold start: behave conservatively rather than abstain entirely
            certified = 0.6 * ph_hat
        promised = min(mag, certified)
        if self.use_risk:
            # value test: expected benefit must exceed expected penalty
            benefit = promised * self.sc.price[t] * dur \
                + promised * self.sc.carbon[t] * dur * 0.05
            shortfall_risk = max(0.0, promised - 0.85 * certified)
            if benefit < self.s.nd_penalty * shortfall_risk * dur:
                promised = 0.0
        if promised < 0.12 * mag:
            promised = 0.0                            # abstain
        return promised


# ----------------------------------------------------------------------------
# 7. Simulation driver and metrics
# ----------------------------------------------------------------------------


def measure_delivery(sc: Scenario, log: pd.DataFrame, ctrl, ref_import,
                     tol=None):
    """Compare promised vs delivered reduction for every commitment."""
    tol = sc.s.deliver_tol if tol is None else tol
    rows = []
    for c in ctrl.commitments:
        t0, dur, req, prom = c["t0"], c["dur"], c["requested"], c["promised"]
        if t0 + dur > len(log):
            continue
        win = slice(t0, t0 + dur)
        delivered = float(np.mean(
            np.maximum(ref_import[win] - log["imp"].values[win], 0.0)))
        rows.append(dict(t0=t0, dur=dur, requested=req, promised=prom,
                         delivered=delivered, decision=c["decision"],
                         shortfall=max(0.0, prom - delivered),
                         covered=bool(delivered >= prom * (1 - tol) - 1e-6)))
    return pd.DataFrame(rows)


def run(sc: Scenario, ctrl_factory, core, feedback=False, ref_import=None,
        respond=True):
    """Run one controller over the whole scenario."""
    plant = Plant(sc)
    ctrl = ctrl_factory(sc, core)
    ctrl.ref_import = ref_import
    ctrl.respond = respond
    for t in range(sc.T):
        ph, pb, pe = ctrl.act(t, plant)
        plant.step(ph, pb, pe)
        if feedback and isinstance(ctrl, CFOController) \
                and ref_import is not None:
            _feedback_step(sc, ctrl, plant, t)
    log = pd.DataFrame(plant.log)
    return ctrl, log


def _feedback_step(sc, ctrl, plant, t):
    """Close the conformal loop: append realised outcomes to calibration."""
    for c in ctrl.commitments:
        if c.get("_done"):
            continue
        t_end = c["t0"] + c["dur"]
        if t + 1 >= t_end:
            log = pd.DataFrame(plant.log)
            ref = ctrl.ref_import[c["t0"]:t_end] \
                if hasattr(ctrl, "ref_import") else None
            if ref is None:
                c["_done"] = True
                continue
            delivered = float(np.mean(np.maximum(
                ref - log["imp"].values[c["t0"]:t_end], 0.0)))
            env = next((e for e in ctrl.envelopes if e["t"] == c["t0"]), None)
            if env is not None:
                ctrl.cert.update(env["phi_hat"], delivered,
                                 ctx=dict(conn_frac=env["conn_frac"],
                                          th_head=env["th_head"],
                                          soc=env["soc"]),
                                 covered=delivered >= c["promised"] * (1 - sc.s.deliver_tol) - 1e-6)
            c["_done"] = True


def summarise(sc, log, deliv, name):
    dt = sc.s.dt
    n_days = sc.s.days
    scale = 365.0 / n_days
    out = dict(controller=name)
    out["CO2_t_per_yr"] = log["co2"].sum() / 1000.0 * scale
    out["cost_per_yr"] = log["cost"].sum() * scale
    out["RE_selfcons_pct"] = 100.0 * (1.0 - log["exp"].sum()
                                      / max(sc.pv.sum(), 1e-9))
    out["peak_kW"] = log["imp"].max()
    out["comfort_viol_h"] = float((log["discomfort"] > 0.15).sum()) * scale
    out["EV_shortfall_kWh"] = log["ev_short"].sum() * scale
    out["bat_cycles"] = log["cyc_bat"].sum() / 2.0 * scale
    out["deg_cost_yr"] = log["deg"].sum() * scale
    if len(deliv):
        prom, deliv_kw = deliv["promised"].values, deliv["delivered"].values
        served = prom > 1e-6
        out["n_requests"] = len(deliv)
        out["n_committed"] = int(served.sum())
        out["abstain_rate_pct"] = 100.0 * (1 - served.mean())
        out["delivery_ratio"] = (deliv_kw[served].sum() / prom[served].sum()
                                 if served.any() else np.nan)
        out["coverage"] = float(deliv.loc[served, "covered"].mean()) \
            if served.any() else np.nan
        out["false_activation_pct"] = 100.0 * float(
            ((deliv["promised"] > 0) & (~deliv["covered"])).mean())
        # missed opportunity: abstained though the request was small
        out["missed_opp_pct"] = 100.0 * float(
            ((deliv["promised"] <= 1e-6)
             & (deliv["requested"] < np.quantile(deliv["requested"], 0.5)))
            .mean())
        pen = np.maximum(prom - deliv_kw, 0).sum() * sc.s.nd_penalty
        rev = deliv_kw.sum() * 0.09
        out["net_service_benefit"] = (rev - pen) * scale
    return out

'''

lib_path = LIBDIR / "cfo_lib.py"
lib_path.write_text(LIB_SOURCE)

import sys
if str(LIBDIR) not in sys.path:
    sys.path.insert(0, str(LIBDIR))

import importlib, cfo_lib
importlib.reload(cfo_lib)
from cfo_lib import (BuildingCfg, FleetCfg, SimCfg, Scenario, Plant, MPCCore,
                     ConformalCertifier, Uncontrolled, TOURule, CarbonRule,
                     MPCController, CFOController, run, measure_delivery,
                     summarise)
print("library written to", lib_path)
print(f"{len(LIB_SOURCE.splitlines())} lines imported")

## 2. Configuration and figure style

In [ ]:
#@title Run configuration { display-mode: "form" }
import time, warnings
import numpy as np, pandas as pd
import matplotlib as mpl, matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

QUICK = False  #@param {type:"boolean"}
DAYS  = 14 if QUICK else 42
SEED  = 20261231  #@param {type:"integer"}

NAVY, STEEL, AMBER = "#123A5E", "#1B5E8C", "#C9930A"
RED, GREEN, GREY   = "#B03A2E", "#1E7A5A", "#8A8F98"
mpl.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 300, "font.size": 9,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "axes.titlesize": 9.5, "axes.titleweight": "bold",
    "legend.frameon": False, "figure.autolayout": True,
})

def savefig(fig, name):
    "Save to Drive and report the path."
    p = FIGS / name
    fig.savefig(p, bbox_inches="tight")
    print("saved", p)
    return p

def savetable(df, name, index=True):
    p = OUT / name
    df.to_csv(p, index=index)
    print("saved", p)
    return p

print(f"DAYS={DAYS}  QUICK={QUICK}  SEED={SEED}")

## 3. Scenario and plant

The EV fleet is described as behavioural clusters with stochastic arrival,
*declared* departure, *actual* departure (an 18% early-departure hazard),
arrival state of charge, required departure state of charge and a
participation propensity. The gap between declared and actual departure is the
dominant source of broken commitments and is what the certification layer must
learn to price.

In [ ]:
scfg = SimCfg(days=DAYS, seed=SEED)
bcfg = BuildingCfg()
fcfg = FleetCfg()

sc   = Scenario(bcfg, fcfg, scfg)
core = MPCCore(bcfg, fcfg, scfg)

print(f"horizon steps       : {sc.T} ({scfg.days} days)")
print(f"ambient temperature : {sc.tamb.min():.1f} .. {sc.tamb.max():.1f} degC")
print(f"PV peak             : {sc.pv.max():.0f} kW of {bcfg.pv_kwp:.0f} kWp")
print(f"grid carbon         : {sc.carbon.min():.2f} .. {sc.carbon.max():.2f} kgCO2/kWh")
print(f"EV fleet            : {sum(fcfg.cluster_sizes)} vehicles in {fcfg.n_clusters} clusters")

### 3.1 Sizing the grid service requests

A flexibility market asks for a reduction relative to the position a site has
**scheduled**, not relative to its unmanaged demand. Sizing requests against an
unmanaged profile makes them unsatisfiable for an already-optimised building —
the controller has typically emptied its evening import already — and the
experiment then measures only infeasibility.

We take the realistic-forecast MPC operating *without any commitment* as the
declared day-ahead schedule and size every request as a fraction of it. The
same request list is then faced by every controller.

In [ ]:
t0 = time.time()
_, schedule = run(sc, lambda s_, c_: MPCController(s_, c_), core, respond=False)
sc.resize_events(schedule["imp"].values)
print(f"{len(sc.events)} requests generated in {time.time()-t0:.0f}s")
mags = np.array([e[2] for e in sc.events])
print(f"magnitude : {mags.min():.0f} .. {mags.max():.0f} kW (mean {mags.mean():.0f})")

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(9.2, 5.0))
d0, d1 = 24*3, 24*6
w = slice(d0, d1); h = np.arange(d0, d1)
ax[0,0].plot(h, sc.tamb[w], color=NAVY, lw=1.4)
ax[0,0].set_ylabel("degC"); ax[0,0].set_title("Ambient temperature")
ax[0,1].fill_between(h, 0, sc.pv[w], color=AMBER, alpha=.55, lw=0)
ax[0,1].plot(h, sc.base_load[w], color=NAVY, lw=1.3, label="base load")
ax[0,1].set_ylabel("kW"); ax[0,1].set_title("PV generation and base load")
ax[0,1].legend(fontsize=7)
ax[1,0].plot(h, sc.carbon[w], color=GREEN, lw=1.4)
ax[1,0].set_ylabel("kgCO$_2$/kWh"); ax[1,0].set_title("Marginal grid carbon intensity")
for c in range(fcfg.n_clusters):
    ax[1,1].step(h, sc.conn[c, w], where="post", lw=1.3, label=f"cluster {c+1}")
ax[1,1].set_ylabel("vehicles"); ax[1,1].set_title("EV fleet availability")
ax[1,1].legend(fontsize=7)
for a in ax.ravel(): a.set_xlabel("hour")
savefig(fig, "fig3_conditions.png"); plt.show()

## 4. Experimental harness

Each controller is run **twice**: once with `respond=False`, giving its declared
schedule, and once facing the grid requests. Delivered flexibility is the mean
import reduction below that controller's *own* declared schedule over the event
window, which isolates the event response from habitual load-shifting.

In [ ]:
def two_pass(scen, mpc_core, factory, feedback=False):
    _, cf = run(scen, factory, mpc_core, respond=False)
    ref = cf["imp"].values
    ctrl, log = run(scen, factory, mpc_core, feedback=feedback,
                    ref_import=ref, respond=True)
    return ctrl, log, measure_delivery(scen, log, ctrl, ref)

def cfo_factory(alpha=0.10, mode="split", **kw):
    def f(s_, c_):
        cert = ConformalCertifier(alpha=alpha, mode=mode,
                                  min_n=s_.s.calib_min, lr=s_.s.adaptive_lr)
        return CFOController(s_, c_, cert, **kw)
    return f

CONTROLLERS = [
    ("B1 uncontrolled",  lambda s_, c_: Uncontrolled(s_),                    False),
    ("B2 TOU rule",      lambda s_, c_: TOURule(s_),                         False),
    ("B3 carbon rule",   lambda s_, c_: CarbonRule(s_),                      False),
    ("B4 MPC perfect",   lambda s_, c_: MPCController(s_, c_, perfect=True), False),
    ("B5 MPC realistic", lambda s_, c_: MPCController(s_, c_),               False),
    ("B6 robust MPC",    lambda s_, c_: MPCController(s_, c_, robust=0.20),  False),
    ("P  CFO",           cfo_factory(alpha=0.10, mode="split"),              True),
]
print("\n".join(n for n, _, _ in CONTROLLERS))

## 5. Main comparison (Table 2 of the manuscript)

In [ ]:
rows, deliveries, logs = [], {}, {}
for name, fac, fb in CONTROLLERS:
    t0 = time.time()
    ctrl, log, d = two_pass(sc, core, fac, feedback=fb)
    rows.append(summarise(sc, log, d, name))
    deliveries[name] = d; logs[name] = log
    savetable(d, f"deliv_{name.split()[0]}.csv", index=False)
    print(f"  {name:18s}  {time.time()-t0:5.1f}s")

main = pd.DataFrame(rows).set_index("controller")
savetable(main, "table2_main_comparison.csv")
main[["CO2_t_per_yr","peak_kW","comfort_viol_h","EV_shortfall_kWh",
      "delivery_ratio","coverage","false_activation_pct",
      "abstain_rate_pct","net_service_benefit"]].round(3)

In [ ]:
srv = main.dropna(subset=["delivery_ratio"])
x = np.arange(len(srv)); labs = [s.replace(" ", "\n", 1) for s in srv.index]
fig, ax = plt.subplots(1, 3, figsize=(10.5, 3.0))
ax[0].bar(x, srv["delivery_ratio"], color=[
    RED if v < .8 else (AMBER if v < .95 else GREEN) for v in srv["delivery_ratio"]])
ax[0].axhline(1.0, color=GREY, ls="--", lw=1); ax[0].set_ylim(0, 1.15)
ax[0].set_title("Flexibility delivery ratio")
ax[1].bar(x, srv["false_activation_pct"], color=NAVY)
ax[1].set_title("False activation rate (%)")
ax[2].bar(x, srv["net_service_benefit"]/1000, color=[
    RED if v < 0 else GREEN for v in srv["net_service_benefit"]])
ax[2].axhline(0, color="k", lw=.8); ax[2].set_title("Net service benefit (k$/yr)")
for a in ax: a.set_xticks(x); a.set_xticklabels(labs, fontsize=7)
savefig(fig, "fig4_reliability.png"); plt.show()

## 6. The reliability–mobility frontier (Figure 5, Table 3)

This is the central experiment. Comparing controllers as *points* is
misleading, because each has a conservatism knob that trades grid reliability
against the resources it consumes to achieve it. Robust MPC has its margin;
CFO has its nominal level $\alpha$. Sweeping both traces two frontiers in the
plane of *delivered reliability* against *unmet EV departure energy*, and the
question becomes which frontier lies closer to the upper-left.

In [ ]:
frontier = []
for r in [0.0, 0.05, 0.10, 0.20, 0.30, 0.40]:
    ctrl, log, d = two_pass(sc, core,
                            lambda s_, c_, rr=r: MPCController(s_, c_, robust=rr))
    m = summarise(sc, log, d, f"robust r={r:.2f}")
    m.update(family="robust MPC", knob=r); frontier.append(m)
    print(f"  robust r={r:.2f}  ratio={m['delivery_ratio']:.3f}  "
          f"EV shortfall={m['EV_shortfall_kWh']:.0f} kWh/yr")

for a in [0.30, 0.20, 0.10, 0.05, 0.02]:
    ctrl, log, d = two_pass(sc, core, cfo_factory(alpha=a), feedback=True)
    m = summarise(sc, log, d, f"CFO alpha={a:.2f}")
    m.update(family="CFO", knob=a); frontier.append(m)
    print(f"  CFO alpha={a:.2f}  ratio={m['delivery_ratio']:.3f}  "
          f"EV shortfall={m['EV_shortfall_kWh']:.0f} kWh/yr")

fr = pd.DataFrame(frontier)
savetable(fr, "table3_frontier.csv", index=False)
fr[["controller","family","knob","delivery_ratio","coverage",
    "EV_shortfall_kWh","comfort_viol_h","net_service_benefit"]].round(3)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9.8, 3.6))
for fam, col, mk in [("robust MPC", AMBER, "v"), ("CFO", STEEL, "o")]:
    g = fr[fr.family == fam].sort_values("EV_shortfall_kWh")
    ax[0].plot(g.EV_shortfall_kWh/1000, g.delivery_ratio, mk+"-",
               color=col, lw=1.8, ms=6, label=fam)
    ax[1].plot(g.EV_shortfall_kWh/1000, g.net_service_benefit/1000, mk+"-",
               color=col, lw=1.8, ms=6, label=fam)
    for _, r_ in g.iterrows():
        ax[0].annotate(f"{r_.knob:g}", (r_.EV_shortfall_kWh/1000, r_.delivery_ratio),
                       fontsize=6.5, xytext=(3, -9), textcoords="offset points",
                       color=col)
ax[0].axhline(1.0, color=GREY, ls="--", lw=1)
ax[0].set_xlabel("unmet EV departure energy (MWh/yr)")
ax[0].set_ylabel("flexibility delivery ratio")
ax[0].set_title("(a) Reliability purchased with mobility")
ax[0].legend(fontsize=7.5, loc="lower right")
ax[1].axhline(0, color="k", lw=.8)
ax[1].set_xlabel("unmet EV departure energy (MWh/yr)")
ax[1].set_ylabel("net service benefit (k$/yr)")
ax[1].set_title("(b) Value purchased with mobility")
ax[1].legend(fontsize=7.5, loc="lower right")
savefig(fig, "fig5_frontier.png"); plt.show()

## 7. Certification quality (Table 4, Figure 6)

In [ ]:
ALPHAS = [0.30, 0.20, 0.10, 0.05, 0.02]
cov_rows = []
for a in ALPHAS:
    ctrl, log, d = two_pass(sc, core, cfo_factory(alpha=a), feedback=True)
    served = d[d.promised > 1e-6]
    post = d.iloc[ctrl.n_commission:]; post = post[post.promised > 1e-6]
    cov_rows.append(dict(alpha=a, nominal=1-a,
        coverage=served.covered.mean() if len(served) else np.nan,
        coverage_post=post.covered.mean() if len(post) else np.nan,
        delivered_kW=served.delivered.sum(), promised_kW=served.promised.sum(),
        abstain_pct=100*(1-len(served)/len(d)), n_committed=len(served)))
    print(f"  alpha={a:.2f}  coverage={cov_rows[-1]['coverage']:.3f}")
cov = pd.DataFrame(cov_rows)
savetable(cov, "table4_certification.csv", index=False)
cov.round(3)

In [ ]:
ref_pts = {nm: (deliveries[nm][deliveries[nm].promised > 0].delivered.sum(),
                deliveries[nm][deliveries[nm].promised > 0].covered.mean())
           for nm in ["B4 MPC perfect", "B5 MPC realistic", "B6 robust MPC"]}

fig, ax = plt.subplots(1, 2, figsize=(9.6, 3.5))
ax[0].plot([.5, 1], [.5, 1], color=GREY, ls="--", lw=1, label="perfect calibration")
ax[0].plot(cov.nominal, cov.coverage, "o-", color=STEEL, lw=1.8, ms=5,
           label="CFO (all events)")
ax[0].plot(cov.nominal, cov.coverage_post, "s-", color=NAVY, lw=1.8, ms=4,
           label="CFO (post-commissioning)")
for nm, (dk, cv) in ref_pts.items():
    c_ = RED if "realistic" in nm else (AMBER if "robust" in nm else GREEN)
    ax[0].axhline(cv, ls=":", lw=1.1, color=c_)
    ax[0].text(.505, cv+.012, nm, fontsize=6.5, color=c_)
ax[0].set_xlabel("nominal coverage  $1-\\alpha$"); ax[0].set_ylabel("empirical coverage")
ax[0].set_title("(a) Calibration of the certified envelope")
ax[0].legend(fontsize=7, loc="lower right")

ax[1].plot(cov.delivered_kW, cov.coverage, "o-", color=STEEL, lw=1.8, ms=5,
           label="CFO frontier ($\\alpha$ sweep)")
for a_, x_, y_ in zip(cov.alpha, cov.delivered_kW, cov.coverage):
    ax[1].annotate(f"$\\alpha$={a_}", (x_, y_), fontsize=6.5,
                   xytext=(3, -9), textcoords="offset points", color=STEEL)
for nm, (dk, cv) in ref_pts.items():
    mk_ = {"B4 MPC perfect":"D","B5 MPC realistic":"^","B6 robust MPC":"v"}[nm]
    c_  = {"B4 MPC perfect":GREEN,"B5 MPC realistic":RED,"B6 robust MPC":AMBER}[nm]
    ax[1].scatter([dk], [cv], marker=mk_, s=55, color=c_, zorder=5, label=nm)
ax[1].set_xlabel("total flexibility delivered (kW-events)")
ax[1].set_ylabel("empirical coverage")
ax[1].set_title("(b) Coverage-value frontier")
ax[1].legend(fontsize=7, loc="lower left")
savefig(fig, "fig6_coverage_value.png"); plt.show()

### 7.1 Promise versus delivery

Each point is one commitment. Points on the diagonal are promises kept. The
clustering of failures near *zero* delivery, rather than along a graduated
shortfall, is the reason a conformal bound on **magnitude** cannot repair
coverage: when a commitment fails, the fleet withdraws entirely rather than
de-rating.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(10.5, 3.3), sharex=True, sharey=True)
for a, nm, col in zip(ax, ["B5 MPC realistic", "B6 robust MPC", "P  CFO"],
                      [RED, AMBER, GREEN]):
    d = deliveries[nm]; s_ = d[d.promised > 1e-6]
    lim = max(s_.promised.max(), s_.delivered.max()) * 1.05
    a.plot([0, lim], [0, lim], color=GREY, ls="--", lw=1)
    a.fill_between([0, lim], [0, lim], [0, 0], color=RED, alpha=.05)
    a.scatter(s_.promised, s_.delivered, s=22, color=col, alpha=.75,
              edgecolor="white", linewidth=.4)
    a.set_title(f"{nm}\n{s_.covered.mean()*100:.0f}% of promises kept", fontsize=8.5)
    a.set_xlabel("promised (kW)")
ax[0].set_ylabel("delivered (kW)")
savefig(fig, "fig7_promise_delivery.png"); plt.show()

## 8. Ablation (Table 5)

In [ ]:
ABL = [
    ("Full CFO",           cfo_factory(0.10, "split")),
    ("- certification",    cfo_factory(0.10, "none")),
    ("- risk term",        cfo_factory(0.10, "split", use_risk=False)),
    ("- information term", cfo_factory(0.10, "split", use_info=False)),
    ("- regime bins",      cfo_factory(0.10, "adaptive")),
]
abl_rows = []
for nm, fac in ABL:
    ctrl, log, d = two_pass(sc, core, fac, feedback=True)
    r_ = summarise(sc, log, d, nm); abl_rows.append(r_)
    print(f"  {nm:20s} ratio={r_['delivery_ratio']:.3f} "
          f"cov={r_['coverage']:.3f} CO2={r_['CO2_t_per_yr']:.1f}")
abl = pd.DataFrame(abl_rows).set_index("controller")
savetable(abl, "table5_ablation.csv")
abl[["CO2_t_per_yr","delivery_ratio","coverage","false_activation_pct",
     "abstain_rate_pct","net_service_benefit"]].round(3)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9.2, 3.0)); xa = np.arange(len(abl))
ax[0].barh(xa, abl["delivery_ratio"], color=[STEEL]+[GREY]*(len(abl)-1))
ax[0].set_yticks(xa); ax[0].set_yticklabels(abl.index, fontsize=7.5)
ax[0].axvline(1.0, color=GREY, ls="--", lw=1); ax[0].invert_yaxis()
ax[0].set_title("Delivery ratio")
ax[1].barh(xa, abl["CO2_t_per_yr"], color=[STEEL]+[GREY]*(len(abl)-1))
ax[1].set_yticks(xa); ax[1].set_yticklabels([]); ax[1].invert_yaxis()
ax[1].set_xlim(abl["CO2_t_per_yr"].min()*.97, abl["CO2_t_per_yr"].max()*1.01)
ax[1].set_title("Operational CO$_2$ (t/yr)")
savefig(fig, "fig8_ablation.png"); plt.show()

## 9. Stress test under forecast error (Table 6)

In [ ]:
SEV = [0.0, 0.5, 1.0, 2.0, 3.0]
stress = []
for sev in SEV:
    scfg_s = SimCfg(days=DAYS, seed=SEED, forecast_severity=sev)
    sc_s   = Scenario(bcfg, fcfg, scfg_s)
    core_s = MPCCore(bcfg, fcfg, scfg_s)
    _, sch_s = run(sc_s, lambda s_, c_: MPCController(s_, c_), core_s, respond=False)
    sc_s.resize_events(sch_s["imp"].values)
    for nm, fac, fb in [("B5 MPC realistic", lambda s_, c_: MPCController(s_, c_), False),
                        ("P  CFO", cfo_factory(0.10), True)]:
        ctrl, log, d = two_pass(sc_s, core_s, fac, feedback=fb)
        r_ = summarise(sc_s, log, d, nm); r_["severity"] = sev; stress.append(r_)
    print(f"  severity {sev}: done")
st = pd.DataFrame(stress)
savetable(st, "table6_stress.csv", index=False)
st.pivot(index="severity", columns="controller",
         values=["delivery_ratio","coverage","abstain_rate_pct"]).round(3)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(10.5, 3.0))
for nm, col, mk in [("B5 MPC realistic", RED, "o"), ("P  CFO", GREEN, "s")]:
    g = st[st.controller == nm]
    ax[0].plot(g.severity, g.delivery_ratio, mk+"-", color=col, label=nm)
    ax[1].plot(g.severity, g.coverage,       mk+"-", color=col, label=nm)
    ax[2].plot(g.severity, g.abstain_rate_pct, mk+"-", color=col, label=nm)
ax[0].axhline(1, color=GREY, ls="--", lw=1)
for a, t in zip(ax, ["Delivery ratio", "Empirical coverage", "Abstention rate (%)"]):
    a.set_title(t); a.set_xlabel("forecast error severity"); a.legend(fontsize=7)
savefig(fig, "fig9_stress.png"); plt.show()

## 10. Case C — cold-climate, winter-peaking regime (Table 7)

> **Known defect.** As parameterised, the heat pump cannot hold the comfort band
> at a mean ambient of 4 °C, so all controllers saturate and the comfort term
> stops discriminating. Re-size `hvac_max`, `cop_heat` or the band before
> interpreting any *thermal* result from this case.

In [ ]:
bcfg_c = BuildingCfg(name="CaseC", winter_peaking=True, mean_temp=4.0,
                     lat_seasonal_amp=15.0, theta_lo=20.0, theta_hi=23.5,
                     theta_set=21.5)
scfg_c = SimCfg(days=DAYS, seed=SEED+11)
sc_c   = Scenario(bcfg_c, fcfg, scfg_c)
core_c = MPCCore(bcfg_c, fcfg, scfg_c)
_, sch_c = run(sc_c, lambda s_, c_: MPCController(s_, c_), core_c, respond=False)
sc_c.resize_events(sch_c["imp"].values)
print(f"Case C: {len(sc_c.events)} requests; ambient "
      f"{sc_c.tamb.min():.1f}..{sc_c.tamb.max():.1f} degC")

rows_c = []
for nm, fac, fb in [("B5 MPC realistic", lambda s_, c_: MPCController(s_, c_), False),
                    ("B6 robust MPC", lambda s_, c_: MPCController(s_, c_, robust=0.20), False),
                    ("P  CFO", cfo_factory(0.10), True)]:
    ctrl, log, d = two_pass(sc_c, core_c, fac, feedback=fb)
    rows_c.append(summarise(sc_c, log, d, nm))
caseC = pd.DataFrame(rows_c).set_index("controller")
savetable(caseC, "table7_caseC.csv")
caseC[["CO2_t_per_yr","peak_kW","comfort_viol_h","EV_shortfall_kWh",
       "delivery_ratio","coverage","abstain_rate_pct"]].round(3)

## 11. Artifact manifest

In [ ]:
summary = pd.concat([main.assign(case="A (temperate)"),
                     caseC.assign(case="C (cold, winter-peaking)")]).reset_index()
savetable(summary, "summary_all_cases.csv", index=False)

print("\nEverything written to:", PROJECT, "\n")
for sub in (OUT, FIGS):
    print(sub.name + "/")
    for p in sorted(sub.iterdir()):
        print(f"   {p.name:38s} {p.stat().st_size/1024:8.1f} kB")

srvA = main.dropna(subset=["delivery_ratio"])
print("\n--- Headline, Case A ---")
for nm in srvA.index:
    r_ = srvA.loc[nm]
    print(f"{nm:18s} delivery {r_.delivery_ratio:5.3f} | coverage {r_.coverage:5.3f} "
          f"| false act {r_.false_activation_pct:5.1f}% | EV short "
          f"{r_.EV_shortfall_kWh:7.0f} kWh/yr | net ${r_.net_service_benefit:,.0f}/yr")

## 12. Before these numbers enter the manuscript

1. **Synthetic data throughout.** Substitute measured weather, marginal-carbon
   and charging-session traces. Each generator sits behind `Scenario._make_*`.
2. **Single seed.** Repeat over at least 10 mobility and forecast seeds and
   report confidence intervals; no difference reported here has been shown to be
   statistically significant.
3. **Certification does not attain nominal coverage.** Empirical coverage fails
   to increase with the nominal level, because delivery failures are binary
   rather than graduated. The fix indicated by Section 7.1 is a two-stage
   construction: a regime classifier predicting whether the system will respond
   at all, with the magnitude quantile applied conditionally. That the
   regime-bin ablation is the most damaging one is direct evidence that the
   regime, not the quantile, carries the information.
4. **Case C is thermally saturated** — see the note in Section 10.
5. **Settlement baseline.** Delivery ratios above 1.0 indicate a controller can
   over-deliver against a schedule its own conservatism has depressed. Tighten
   the baseline definition before quoting those figures.
6. **No RL baseline** is implemented; do not claim one.